# 🚀 colab-llm-host

**Turn a free Google Colab session into your own LLM backend — no GPU, no cost, no local setup.**

Run open-source models (Gemma, Qwen, Llama) on Colab's free T4 GPU and talk to them from any device through a secure tunnel.

---

### How to use
1. Set the runtime to GPU → **Runtime → Change runtime type → T4 GPU → Save**
2. Paste your free **ngrok token** in the cell below (get it at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken))
3. **Runtime → Run all**
4. Copy the public URL that prints at the bottom — that's your backend

> ⚠️ The session is temporary. When Colab times out, re-run this notebook to get a fresh URL.

## Step 1 — Check the GPU
Confirms a GPU is attached. If this errors, switch the runtime to **T4 GPU** and run again.

In [ ]:
!nvidia-smi -L || print('⚠️ No GPU found — go to Runtime > Change runtime type > T4 GPU, then re-run.')

## Step 2 — Install Ollama
Installs `zstd` (needed to unpack the download) and then Ollama itself. Takes ~1–2 minutes.

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
print('\n✅ Ollama installed')

## Step 3 — Start the Ollama server
Runs Ollama in the background, bound to `0.0.0.0` so the tunnel can reach it.

In [ ]:
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
print('✅ Ollama server running on port 11434')

## Step 4 — Pick and download a model
Choose from the dropdown. Smaller = faster.

| Model | Size | Speed | Best for |
|-------|------|-------|----------|
| `gemma3:1b` | ~0.8 GB | ⚡ fastest | quick, simple questions |
| `gemma3:4b` | ~3 GB | fast | balanced everyday use |
| `llama3.2:3b` | ~2 GB | fast | general all-rounder |
| `qwen3:8b` | ~5 GB | slower | reasoning, harder tasks |

> 💡 `qwen3` and `deepseek-r1` "think" before answering, so they feel slower. For instant replies, pick a Gemma or Llama model.

In [ ]:
MODEL = 'gemma3:4b'  # @param ["gemma3:1b", "gemma3:4b", "llama3.2:3b", "qwen3:8b"]

print(f'📥 Downloading {MODEL} ...')
!ollama pull {MODEL}
print(f'\n✅ {MODEL} ready')

## Step 5 — Open the tunnel
Paste your free ngrok token below. This creates the public URL your other devices connect to.

In [ ]:
NGROK_TOKEN = ''  # @param {type:"string"}

!pip install -q pyngrok
from pyngrok import ngrok

if not NGROK_TOKEN:
    raise ValueError('Paste your ngrok token above. Get one free at https://dashboard.ngrok.com/get-started/your-authtoken')

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(11434, host_header='localhost:11434')

print('=' * 60)
print('🌐 Your backend is live at:')
print('   ', tunnel.public_url)
print('=' * 60)
print('Copy this URL — you\'ll use it from your other device.')

## Step 6 — Test it right here
A quick sanity check that the whole pipeline works before you leave Colab.

In [ ]:
import requests

r = requests.post(
    tunnel.public_url + '/api/generate',
    json={'model': MODEL, 'prompt': 'Say hello in one short sentence.', 'stream': False},
    headers={'ngrok-skip-browser-warning': 'true'},
    timeout=180,
)
print('🤖', r.json().get('response', r.text).strip())

---
## ✅ Done! Now use it from your PC

Keep this Colab tab **open** (closing it kills the server).

**Windows PowerShell** — paste this, replace the URL with yours, then just type a question each time:
```powershell
$q = Read-Host "Ask"
$body = @{ model = "gemma3:4b"; prompt = $q; stream = $false } | ConvertTo-Json
(Invoke-RestMethod -Uri "https://YOUR-URL.ngrok-free.dev/api/generate" -Method Post -Body $body -ContentType "application/json").response
```

**Mac / Linux terminal:**
```bash
curl https://YOUR-URL.ngrok-free.dev/api/generate -d '{"model":"gemma3:4b","prompt":"what is india","stream":false}'
```

**Prefer a chat window?** Point any OpenAI-compatible app (Open WebUI, Continue, etc.) at your URL as the base address.

---
*Made with colab-llm-host · free LLMs, zero hardware.*